# 01 - Ingest public data (SPEC section 9.1)
Lands every public source that has a bulk download into the raw volume, unchanged, as a dated snapshot:
`/Volumes/<catalog>/<schema>/<raw_volume>/public/<source_id>/snapshot=<yyyymmdd>/<file>`

Each snapshot writes a `_manifest.json` (url, bytes, sha256, licence, `provenance=public`). Snapshots are never overwritten, so weekly runs build the status-change history the collapse model needs.

URLs are resolved at run time (data.gov.au CKAN API, or by scraping the publisher page), so nothing goes stale when publishers rename monthly files. Web-only sources (P7, P8, P9, P15, P16) are covered by the synthetic layer in notebook 02.

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("raw_volume", "raw")
dbutils.widgets.text("legal_corpus_max_docs", "2000")
dbutils.widgets.text("include_abn_bulk_extract", "true")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RAW_VOLUME = dbutils.widgets.get("raw_volume")
LEGAL_MAX_DOCS = int(dbutils.widgets.get("legal_corpus_max_docs"))
INCLUDE_ABN = dbutils.widgets.get("include_abn_bulk_extract").lower() == "true"
assert CATALOG and SCHEMA, "catalog and schema parameters are required"

import datetime as dt
RAW_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_VOLUME}"
SNAPSHOT = dt.date.today().strftime("%Y%m%d")
print(RAW_ROOT, SNAPSHOT)

In [0]:
# kind: ckan   -> data.gov.au package id + preferred formats (first match wins per resource name)
#       scrape -> publisher page + regex on linked file names
#       direct -> fixed URL
#       hf     -> Hugging Face JSONL, streamed and filtered
CKAN_API = "https://data.gov.au/data/api/3/action/package_show"
SOURCES = [
    # P1-P5 ASIC registers
    {"id": "p01_asic_companies",        "kind": "ckan", "package": "asic-companies",              "formats": ["ZIP"], "licence": "CC BY 3.0 AU"},
    {"id": "p02_asic_business_names",   "kind": "ckan", "package": "asic-business-names",         "formats": ["ZIP"], "licence": "CC BY 3.0 AU"},
    {"id": "p03_asic_afs_licensees",    "kind": "ckan", "package": "asic-afs-licensee",           "formats": ["TSV"], "licence": "CC BY 3.0 AU"},
    {"id": "p03_asic_credit_licensees", "kind": "ckan", "package": "asic-credit-licensee",        "formats": ["TSV"], "licence": "CC BY 3.0 AU", "name_excludes": "National Map"},
    {"id": "p04_asic_banned_persons",   "kind": "ckan", "package": "asic-banned-disqualified-per", "formats": ["TSV"], "licence": "CC BY 3.0 AU"},
    {"id": "p04_asic_banned_orgs",      "kind": "ckan", "package": "asic-banned-disqualified-org", "formats": ["TSV"], "licence": "CC BY 3.0 AU"},
    {"id": "p05_asic_liquidators",      "kind": "ckan", "package": "asic-liquidator",             "formats": ["TSV"], "licence": "CC BY 3.0 AU"},
    # P6 ASIC insolvency statistics (series 1, 2 and 3 workbooks)
    {"id": "p06_asic_insolvency_stats", "kind": "scrape",
     "page": "https://asic.gov.au/regulatory-resources/find-a-document/statistics/insolvency-statistics/",
     "pattern": r"insolvency-statistics-series", "ext": (".xlsx",), "licence": "CC BY 4.0"},
    # P10 Open Australian Legal Corpus (ASIC v ... judgments only)
    {"id": "p10_open_au_legal_corpus",  "kind": "hf",
     "url": "https://huggingface.co/datasets/isaacus/open-australian-legal-corpus/resolve/main/corpus.jsonl",
     "match": r"Australian Securities (and|&) Investments Commission v|\bASIC v\b", "licence": "See corpus LICENCE.md (per-source licences)"},
    # P11 ABN Bulk Extract (optional, ~1 GB)
    {"id": "p11_abn_bulk_extract",      "kind": "ckan", "package": "abn-bulk-extract", "formats": ["ZIP"], "licence": "CC BY 3.0 AU", "enabled": INCLUDE_ABN},
    # P12 ATO Corporate Tax Transparency (all years)
    {"id": "p12_ato_tax_transparency",  "kind": "ckan", "package": "corporate-transparency", "formats": ["XLSX", "EXCEL (.XLSX)"], "licence": "CC BY 2.5 AU", "all": True},
    # P13 APRA superannuation statistics
    {"id": "p13_apra_super_stats",      "kind": "scrape", "page": "https://www.apra.gov.au/quarterly-superannuation-statistics",
     "pattern": r".", "ext": (".xlsx",), "licence": "CC BY 3.0 AU"},
    # P14 APRA monthly ADI statistics
    {"id": "p14_apra_adi_stats",        "kind": "scrape", "page": "https://www.apra.gov.au/monthly-authorised-deposit-taking-institution-statistics",
     "pattern": r"Monthly%20authorised", "ext": (".xlsx",), "licence": "CC BY 3.0 AU"},
    # P17 DFAT consolidated sanctions list
    {"id": "p17_dfat_sanctions",        "kind": "direct",
     "url": "https://www.dfat.gov.au/sites/default/files/Australian_Sanctions_Consolidated_List.xlsx", "licence": "CC BY 4.0"},
    # P18 ACNC charity register
    {"id": "p18_acnc_register",         "kind": "ckan", "package": "acnc-register", "formats": ["CSV"], "licence": "CC BY 3.0 AU"},
]
print(len(SOURCES), "sources")

In [0]:
import os, re, json, hashlib, zipfile, shutil, time, urllib.parse, requests

HEADERS = {"User-Agent": "Mozilla/5.0 (asic-case-management ingest)"}

def resolve(src):
    """Return a list of (url, filename) for a source."""
    if src["kind"] == "ckan":
        res = requests.get(CKAN_API, params={"id": src["package"]}, timeout=60).json()["result"]["resources"]
        fmts = [f.upper() for f in src["formats"]]
        hits = [r for r in res if (r.get("format") or "").upper() in fmts
                and not (src.get("name_excludes") and src["name_excludes"] in (r.get("name") or ""))]
        if not src.get("all"):
            hits = hits[:1] if src["package"] != "abn-bulk-extract" else hits
        return [(r["url"], os.path.basename(urllib.parse.urlparse(r["url"]).path)) for r in hits]
    if src["kind"] == "scrape":
        html = requests.get(src["page"], headers=HEADERS, timeout=60).text
        links = sorted(set(re.findall(r'href="([^"]+)"', html)))
        out = []
        for l in links:
            if l.lower().endswith(src["ext"]) and re.search(src["pattern"], l, re.I):
                url = urllib.parse.urljoin(src["page"], l)
                out.append((url, urllib.parse.unquote(os.path.basename(urllib.parse.urlparse(url).path))))
        return out
    if src["kind"] in ("direct", "hf"):
        return [(src["url"], os.path.basename(urllib.parse.urlparse(src["url"]).path))]
    raise ValueError(src["kind"])

def download(url, dest, retries=3):
    """Stream to a local temp file, then copy into the volume. Returns (bytes, sha256)."""
    tmp = f"/tmp/{hashlib.md5(url.encode()).hexdigest()}"
    for attempt in range(retries):
        try:
            h = hashlib.sha256(); n = 0
            with requests.get(url, headers=HEADERS, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(8 << 20):
                        f.write(chunk); h.update(chunk); n += len(chunk)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copyfile(tmp, dest); os.remove(tmp)
            return n, h.hexdigest()
        except Exception as e:
            print(f"  retry {attempt+1} {url}: {e}"); time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"failed {url}")

def unzip_into(zip_path, out_dir):
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            if m.is_dir():
                continue
            target = os.path.join(out_dir, os.path.basename(m.filename))
            with z.open(m) as s, open(target, "wb") as d:
                shutil.copyfileobj(s, d, 8 << 20)
    os.remove(zip_path)

In [0]:
def ingest_legal_corpus(src, out_dir, max_docs, max_minutes=45):
    """Stream the 9 GB JSONL and keep only ASIC v ... decisions (by citation) up to max_docs."""
    rx = re.compile(src["match"])
    os.makedirs(out_dir, exist_ok=True)
    tmp = "/tmp/asic_judgments.jsonl"
    kept = scanned = 0; start = time.time()
    pos, attempts, done, complete = 0, 0, False, False  # pos = bytes of fully processed lines (for Range resume)
    with open(tmp, "w") as f:
        while not done and attempts < 6:
            headers = dict(HEADERS, **({"Range": f"bytes={pos}-"} if pos else {}))
            buf = b""
            try:
                with requests.get(src["url"], headers=headers, stream=True, timeout=600) as r:
                    r.raise_for_status()
                    for chunk in r.iter_content(8 << 20):
                        *lines, buf = (buf + chunk).split(b"\n")
                        for raw in lines:
                            pos += len(raw) + 1
                            if not raw:
                                continue
                            scanned += 1
                            head = raw[:2000]
                            if b"Securities" not in head and b"ASIC v" not in head:  # cheap prefilter
                                continue
                            doc = json.loads(raw.decode("utf-8", "replace"))
                            if rx.search(doc.get("citation") or ""):
                                doc["provenance"] = "public"
                                f.write(json.dumps(doc) + "\n"); kept += 1
                            if kept >= max_docs or (time.time() - start) > max_minutes * 60:
                                done = True
                                break
                        if done:
                            break
                complete = not done
                done = True
            except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as e:
                attempts += 1
                print(f"  connection dropped at byte {pos:,} ({e.__class__.__name__}); resuming, attempt {attempts}")
                time.sleep(10 * attempts)
    dest = os.path.join(out_dir, "asic_judgments.jsonl")
    shutil.copyfile(tmp, dest)
    print(f"  scanned {scanned:,} docs, kept {kept:,} ASIC v ... decisions, full scan={complete}")
    return [{"url": src["url"], "file": dest, "docs_kept": kept, "docs_scanned": scanned, "full_scan": complete}]

In [0]:
results = []
for src in SOURCES:
    if src.get("enabled", True) is False:
        print("skip", src["id"]); continue
    out_dir = f"{RAW_ROOT}/public/{src['id']}/snapshot={SNAPSHOT}"
    if os.path.exists(f"{out_dir}/_manifest.json"):
        print("exists", out_dir); continue
    print("==>", src["id"])
    try:
        if src["kind"] == "hf":
            files = ingest_legal_corpus(src, out_dir, LEGAL_MAX_DOCS)
        else:
            files = []
            for url, name in resolve(src):
                dest = f"{out_dir}/{name}"
                n, sha = download(url, dest)
                if name.lower().endswith(".zip"):
                    unzip_into(dest, out_dir)
                files.append({"url": url, "file": name, "bytes": n, "sha256": sha})
                print(f"  {name} {n/1e6:,.1f} MB")
        manifest = {"source_id": src["id"], "snapshot": SNAPSHOT, "licence": src["licence"],
                    "provenance": "public", "loaded_at": dt.datetime.utcnow().isoformat() + "Z", "files": files}
        with open(f"{out_dir}/_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        results.append((src["id"], "ok", len(files)))
    except Exception as e:
        results.append((src["id"], f"FAILED: {e}", 0))
        print("  FAILED", e)

display(spark.createDataFrame(results, "source_id string, status string, files int"))

In [0]:
# P1 (entity master + collapse labels) and P4 (enforcement history) are mandatory; others degrade gracefully.
CORE = {"p01_asic_companies", "p04_asic_banned_persons"}
failed_core = [s for s, status, _ in results if s in CORE and status != "ok"]
assert not failed_core, f"Core public sources failed: {failed_core}"
for s in CORE:
    assert any(os.scandir(f"{RAW_ROOT}/public/{s}")), f"no snapshot for {s}"